In [ ]:
import os
import numpy as np
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from typing import List, Optional, Dict, Any
import warnings
warnings.filterwarnings('ignore')

class LinearRegressionPipeline:
    """
    ========================================================
    GENERIC PYSPARK LINEAR REGRESSION PIPELINE TEMPLATE
    ========================================================
    
    COMPLETE PRODUCTION-READY template for ANY regression problem.
    Predicts continuous numeric targets like house prices, sales, etc.
    
    KEY FEATURES:
    • Automatic CSV loading + schema inspection
    • Numeric + categorical feature handling
    • Full preprocessing pipeline (encoding → scaling → assembly)
    • Train/test split (80/20) with reproducible seed
    • Regression metrics: MAE, RMSE, R²
    • Production-ready prediction pipeline
    • Extensive logging + null detection
    • YOUR HOUSING DATASET READY!
    """

    def __init__(
        self, 
        app_name: str = 'LinearRegressionPipeline',
        # ==============================================
        # CONFIGURATION - Set your target column name
        # ==============================================
        label_col: str = 'target',           # YOUR TARGET (median_house_value)
        features_col: str = 'features',      # Final feature vector column
        prediction_col: str = 'prediction'   # Model output column
    ):
        """
        Initialize Linear Regression Pipeline for continuous target prediction.
        
        Args:
            app_name: Spark application name
            label_col: Target column (e.g., 'median_house_value')
            features_col: Final assembled features column
            prediction_col: Prediction output column name
        """
        print("🚀 Initializing Linear Regression Pipeline...")
        
        # Initialize Spark session (connects to local/cluster)
        self.spark = SparkSession.builder.appName(app_name).getOrCreate()
        
        # Store configuration
        self.label_col = label_col
        self.features_col = features_col
        self.prediction_col = prediction_col
        
        # Feature configuration (user sets these)
        self.categorical_cols = []           # Categorical columns to encode
        self.numeric_cols = []              # Numeric columns (pass-through)
        self.feature_columns = []           # Final feature list
        
        # Data pipeline stages
        self.data = None                    # Raw input DataFrame
        self.training_data = None           # 80% training split
        self.testing_data = None            # 20% test split
        
        # Trained pipeline components
        self.model = None                   # Final LinearRegression model
        self.pipeline_models = {}           # Preprocessing models (encoders, scaler)
        
        # Results storage
        self.metrics = {}                   # MAE, RMSE, R² scores
        
        print("✅ Linear Regression Pipeline ready!")
    
    def load_data(self, csv_path: str, header: bool = True, infer_schema: bool = True):
        """
        Load CSV data into Spark DataFrame with automatic schema detection.
        
        Args:
            csv_path: Local/HDFS/S3 path to your CSV file
            header: CSV has column names in first row
            infer_schema: Auto-detect column types (double, string, etc)
        """
        print(f"\n📂 Loading data: {csv_path}")
        
        self.data = self.spark.read.csv(csv_path, header=header, inferSchema=infer_schema)
        row_count = self.data.count()
        print(f"✅ Loaded {row_count:,} rows × {len(self.data.columns)} columns")
        
        # Show schema and sample
        self._print_schema()
        print("\n📋 Sample rows:")
        self.data.show(5, truncate=False)
    
    def set_feature_columns(self, numeric_cols: List[str], categorical_cols: List[str]):
        """
        Define which columns are features for the model.
        
        Args:
            numeric_cols: Continuous features (no transformation needed)
            categorical_cols: String categories to encode
            
        IMPORTANT: Target column NOT included here!
        """
        print(f"\n🔧 Setting up features...")
        print(f"   📊 Numeric ({len(numeric_cols)}): {numeric_cols}")
        print(f"   🏷️  Categorical ({len(categorical_cols)}): {categorical_cols}")
        
        self.numeric_cols = numeric_cols
        self.categorical_cols = categorical_cols
        
        # Final features = numeric + encoded categorical vectors
        self.feature_columns = numeric_cols + [f"{cat}_vec" for cat in categorical_cols]
        print(f"✅ {len(self.feature_columns)} total features ready")
    
    def clean_data(self, drop_cols: List[str] = None, subset_cols: List[str] = None):
        """
        Remove unwanted columns and rows with missing values.
        
        Args:
            drop_cols: Columns to DROP entirely (feature selection)
            subset_cols: Rows with NULLs in these columns get dropped
            
        PRO TIP: Always include target column in subset_cols!
        """
        print(f"\n🧹 Cleaning dataset...")
        
        # Remove unwanted columns
        if drop_cols:
            print(f"   ❌ Dropping: {drop_cols}")
            self.data = self.data.drop(*drop_cols)
        
        # Remove rows with critical missing values
        if subset_cols:
            print(f"   🔍 Checking nulls: {subset_cols}")
            before = self.data.count()
            self.data = self.data.dropna(subset=subset_cols)
            after = self.data.count()
            print(f"   📉 Rows: {before:,} → {after:,} ({after/before*100:.1f}%)")
        
        self._check_nulls()
    
    def preprocess(self):
        """
        FULL FEATURE ENGINEERING PIPELINE:
        1. Train/Test split (80/20, reproducible)
        2. Categorical encoding (StringIndexer → OneHotEncoder)
        3. Vector assembly (all features → single vector)
        4. Standardization (mean=0, std=1)
        """
        print(f"\n🔄 Feature engineering pipeline...")
        
        # Split data (fixed seed = reproducible results)
        print("   1️⃣ Train/Test split (80/20)...")
        self.training_data, self.testing_data = self.data.randomSplit([0.8, 0.2], seed=42)
        print(f"      📚 Training: {self.training_data.count():,} rows")
        print(f"      🧪 Testing:  {self.testing_data.count():,} rows")
        
        # ========================================
        # CATEGORICAL ENCODING PIPELINE
        # ========================================
        print("   2️⃣ Encoding categorical variables...")
        for cat_col in self.categorical_cols:
            print(f"      🏷️  {cat_col}")
            
            # String → numeric index
            indexer = StringIndexer(inputCol=cat_col, outputCol=f'{cat_col}_index')
            indexer_model = indexer.fit(self.training_data)
            
            # Index → one-hot binary vector
            encoder = OneHotEncoder(inputCol=f'{cat_col}_index', outputCol=f'{cat_col}_vec', dropLast=False)
            encoder_model = encoder.fit(indexer_model.transform(self.training_data))
            
            # Save models for new data prediction
            self.pipeline_models[f'{cat_col}_indexer'] = indexer_model
            self.pipeline_models[f'{cat_col}_encoder'] = encoder_model
            
            # Transform training data
            self.training_data = encoder_model.transform(indexer_model.transform(self.training_data))
        
        # ========================================
        # FEATURE VECTOR CREATION
        # ========================================
        print("   3️⃣ Assembling feature vectors...")
        assembler = VectorAssembler(inputCols=self.feature_columns, outputCol='unscaled_features')
        self.training_data = assembler.transform(self.training_data)
        self.pipeline_models['assembler'] = assembler
        
        # Scale features (CRITICAL for Linear Regression!)
        print("   4️⃣ Standardizing features...")
        scaler = StandardScaler(
            inputCol='unscaled_features',
            outputCol=self.features_col,
            withMean=True, withStd=True  # Z-score normalization
        )
        scaler_model = scaler.fit(self.training_data)
        self.training_data = scaler_model.transform(self.training_data)
        self.pipeline_models['scaler'] = scaler_model
        
        print(f"✅ Preprocessing done! {len(self.feature_columns)}-D feature vectors")
    
    def train(self, max_iter: int = 100, reg_param: float = 0.01):
        """
        Train Linear Regression model with L2 regularization.
        
        Args:
            max_iter: Maximum optimization iterations
            reg_param: Regularization strength (prevents overfitting)
        """
        print(f"\n🎯 Training Linear Regression...")
        print(f"   🔧 max_iter={max_iter}, reg_param={reg_param}")
        
        lr = LinearRegression(
            featuresCol=self.features_col,
            labelCol=self.label_col,
            predictionCol=self.prediction_col,
            maxIter=max_iter,
            regParam=reg_param
        )
        
        self.model = lr.fit(self.training_data)
        print("✅ Model trained successfully!")
    
    def preprocess_test_data(self):
        """Apply identical preprocessing to test data."""
        test_data = self.testing_data
        
        # Same categorical pipeline as training
        for cat_col in self.categorical_cols:
            test_data = self.pipeline_models[f'{cat_col}_indexer'].transform(test_data)
            test_data = self.pipeline_models[f'{cat_col}_encoder'].transform(test_data)
        
        test_data = self.pipeline_models['assembler'].transform(test_data)
        test_data = self.pipeline_models['scaler'].transform(test_data)
        return test_data
    
    def evaluate(self):
        """Calculate regression metrics: MAE, RMSE, R²"""
        print(f"\n📊 Model evaluation...")
        
        test_data = self.preprocess_test_data()
        predictions = self.model.transform(test_data)
        
        # Industry standard regression metrics
        mae = RegressionEvaluator(labelCol=self.label_col, predictionCol=self.prediction_col, metricName='mae').evaluate(predictions)
        rmse = RegressionEvaluator(labelCol=self.label_col, predictionCol=self.prediction_col, metricName='rmse').evaluate(predictions)
        r2 = RegressionEvaluator(labelCol=self.label_col, predictionCol=self.prediction_col, metricName='r2').evaluate(predictions)
        
        self.metrics = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
        
        print("\n" + "="*60)
        print("🏆 FINAL REGRESSION PERFORMANCE")
        print("="*60)
        print(f"Mean Absolute Error (MAE)   : ${mae:,.0f}")
        print(f"Root Mean Squared Error     : ${rmse:,.0f}")
        print(f"R-squared (Explained Var)   : {r2:.4f}")
        print("="*60)
        
        return self.metrics
    
    def predict(self, new_data):
        """Predict on completely new data."""
        if self.model is None:
            raise ValueError("❌ Train model first!")
        
        print("🔮 Predicting new data...")
        processed = new_data
        
        # Full preprocessing pipeline
        for cat_col in self.categorical_cols:
            processed = self.pipeline_models[f'{cat_col}_indexer'].transform(processed)
            processed = self.pipeline_models[f'{cat_col}_encoder'].transform(processed)
        
        processed = self.pipeline_models['assembler'].transform(processed)
        processed = self.pipeline_models['scaler'].transform(processed)
        
        return self.model.transform(processed)
    
    def fit_predict(self, csv_path: str, 
                    numeric_cols: List[str], 
                    categorical_cols: List[str],
                    drop_null_subset: List[str] = None):
        """ONE-LINE COMPLETE PIPELINE EXECUTION!"""
        print("\n" + "="*80)
        print("🚀 LINEAR REGRESSION PIPELINE - FULL EXECUTION")
        print("="*80)
        
        self.load_data(csv_path)
        self.set_feature_columns(numeric_cols, categorical_cols)
        self.clean_data(subset_cols=drop_null_subset)
        self.preprocess()
        self.train()
        
        metrics = self.evaluate()
        print("\n🎊 PIPELINE COMPLETE!")
        return metrics
    
    # ========================================
    # UTILITY METHODS
    # ========================================
    def _print_schema(self):
        print("\n📋 Dataset Schema:")
        self.data.printSchema()
    
    def _check_nulls(self):
        null_counts = self.data.select([
            F.count(F.when(F.col(c).isNull(), 1)).alias(c) 
            for c in self.data.columns
        ]).collect()[0]
        
        print("🔍 Null values:")
        has_nulls = False
        for c in self.data.columns:
            count = getattr(null_counts, c)
            if count > 0:
                pct = (count / self.data.count()) * 100
                print(f"   {c:>20}: {count:,} ({pct:5.1f}%)")
                has_nulls = True
        if not has_nulls:
            print("   ✅ Clean dataset!")
    
    def stop(self):
        print("\n🛑 Stopping Spark...")
        self.spark.stop()



In [ ]:
# ========================================================
# TESTING AGAINST house_prices.csv
# ========================================================

if __name__ == "__main__":
    print("🏠 HOUSING PRICE PREDICTION EXAMPLE")
    
    # YOUR EXACT COLUMNS FROM THE SCHEMA!
    house_predictor = LinearRegressionPipeline(
        label_col='median_house_value',  # YOUR TARGET
        app_name='HousingPricePredictor'
    )
    
    # ONE LINE - YOUR EXACT DATASET!
    results = house_predictor.fit_predict(
        csv_path='work/housing_prices.csv',
        
        # YOUR NUMERIC COLUMNS
        numeric_cols=[
            'longitude', 'latitude',
            'housing_median_age', 'total_rooms', 
            'total_bedrooms', 'population',
            'households', 'median_income'
        ],
        
        # YOUR CATEGORICAL COLUMN
        categorical_cols=['ocean_proximity'],
        
        # Handle your nulls
        drop_null_subset=['total_bedrooms']
    )
    
    # Show sample predictions
    print("\n🏠 SAMPLE HOUSE PRICE PREDICTIONS:")
    test_data = house_predictor.preprocess_test_data()
    predictions = house_predictor.model.transform(test_data)
    predictions.select(
        'longitude', 'latitude', 
        'median_house_value', 
        'prediction'
    ).show(5)
    
    house_predictor.stop()
